# Federated Learning Experiment t-Test Summary Analyzer

This notebook parses the experimental results across 5 seed Excel files, calculates paired t-tests (comparing other algorithms to the proposed `unicsl_static` method), and outputs a single Excel summary workbook with multiple sheets, along with a unified HTML report.

In [9]:
# Install dependencies if needed
# !pip install pandas numpy scipy openpyxl

In [10]:
import os
import re
import zipfile
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel
from scipy import stats

## 1. Helper Functions for XML Excel Parsing

In [11]:
def col_to_idx(col_str):
    idx = 0
    for char in col_str:
        if 'A' <= char <= 'Z':
            idx = idx * 26 + (ord(char) - ord('A') + 1)
    return idx - 1

def parse_cell_ref(ref):
    match = re.match(r'([A-Z]+)(\d+)', ref)
    if match:
        col_str, row_str = match.groups()
        return col_to_idx(col_str), int(row_str) - 1
    return None

def parse_sheet_data(z, full_path, shared_strings):
    try:
        sheet_xml = z.read(full_path)
        sheet_root = ET.fromstring(sheet_xml)
        grid = {}
        max_r, max_c = 0, 0
        for row_elem in sheet_root.iter():
            if row_elem.tag.endswith('row'):
                r_idx = int(row_elem.attrib.get('r', 1)) - 1
                for cell_elem in row_elem.iter():
                    if cell_elem.tag.endswith('c'):
                        ref = cell_elem.attrib.get('r')
                        t_type = cell_elem.attrib.get('t')
                        
                        v_elem = None
                        for sub in cell_elem:
                            if sub.tag.endswith('v'):
                                v_elem = sub
                                break
                        
                        val = ''
                        if v_elem is not None:
                            val = v_elem.text or ''
                        
                        if t_type == 's' and val:
                            try:
                                val = shared_strings[int(val)]
                            except (IndexError, ValueError):
                                pass
                        
                        if ref:
                            c_idx, r_idx_parsed = parse_cell_ref(ref)
                            grid[(r_idx_parsed, c_idx)] = val
                            max_r = max(max_r, r_idx_parsed)
                            max_c = max(max_c, c_idx)
        
        rows = []
        for r in range(max_r + 1):
            row_vals = [grid.get((r, c), '') for c in range(max_c + 1)]
            rows.append(row_vals)
        return rows
    except Exception as e:
        print(f"Error parsing sheet {full_path}: {e}")
        return []

## 2. Data Collection Logic

In [12]:
def collect_experiment_data(folder):
    xlsx_files = sorted([f for f in os.listdir(folder) if f.endswith('.xlsx') and 'comparison_results' not in f])
    
    data = {
        scen: {
            met: {
                ds: {
                    algo: {} for algo in ['fedavg', 'fedprox', 'scaffold', 'feddyn', 'fedopt', 'pfedme', 'unicsl_static']
                } for ds in ['cifar10', 'fashionmnist', 'mnist']
            } for met in ['accuracy_final', 'accuracy_average', 'time', 'memory']
        } for scen in ['iid', 'non_iid', 'non_iid_corrupted']
    }
    
    for filename in xlsx_files:
        match = re.match(r'(\d+)_experimental_results\.xlsx', filename)
        if not match:
            continue
        seed = int(match.group(1))
        file_path = os.path.join(folder, filename)
        
        with zipfile.ZipFile(file_path, 'r') as z:
            shared_strings = []
            try:
                ss_xml = z.read('xl/sharedStrings.xml')
                root = ET.fromstring(ss_xml)
                for si in root:
                    text = "".join(t.text for t in si.iter() if t.tag.endswith('t') and t.text)
                    shared_strings.append(text)
            except KeyError:
                pass
                
            rels = {}
            try:
                rels_xml = z.read('xl/_rels/workbook.xml.rels')
                rels_root = ET.fromstring(rels_xml)
                for rel in rels_root:
                    rels[rel.attrib.get('Id')] = rel.attrib.get('Target')
            except Exception:
                pass
                
            wb_xml = z.read('xl/workbook.xml')
            wb_root = ET.fromstring(wb_xml)
            sheets = {}
            for elem in wb_root.iter():
                if elem.tag.endswith('sheet'):
                    r_id = None
                    for k, v in elem.attrib.items():
                        if k.endswith('id'):
                            r_id = v
                    sheets[elem.attrib.get('name')] = r_id
            
            for scen in ['iid', 'non_iid', 'non_iid_corrupted']:
                # 1. Process Accuracy
                sheet_name = scen
                r_id = sheets.get(sheet_name)
                if r_id:
                    target_path = rels.get(r_id)
                    full_path = f"xl/{target_path}" if target_path and not target_path.startswith('xl/') else f"xl/worksheets/sheet{r_id}.xml"
                    rows = parse_sheet_data(z, full_path, shared_strings)
                    if rows:
                        header = [c.strip() for c in rows[0]]
                        dataset_idx = next((i for i, c in enumerate(header) if 'dataset' in c.lower()), -1)
                        round_idx = next((i for i, c in enumerate(header) if 'round' in c.lower()), -1)
                        
                        algos_present = [algo for algo in ['fedavg', 'fedprox', 'scaffold', 'feddyn', 'fedopt', 'pfedme', 'unicsl_static'] if algo in header]
                        algo_indices = {algo: header.index(algo) for algo in algos_present}
                        
                        for row in rows[1:]:
                            if not row or len(row) <= max(dataset_idx, round_idx):
                                continue
                            ds = str(row[dataset_idx]).strip().lower()
                            round_val = str(row[round_idx]).strip()
                            
                            if ds not in ['cifar10', 'fashionmnist', 'mnist']:
                                continue
                                
                            is_final_round = False
                            is_avg_round = False
                            
                            if round_val.lower() == 'average':
                                is_avg_round = True
                            else:
                                try:
                                    if int(float(round_val)) == 50:
                                        is_final_round = True
                                except ValueError:
                                    pass
                                    
                            if is_final_round or is_avg_round:
                                metric_key = 'accuracy_final' if is_final_round else 'accuracy_average'
                                for algo in algos_present:
                                    idx = algo_indices[algo]
                                    if idx < len(row) and row[idx] != '':
                                        try:
                                            val = float(row[idx])
                                            data[scen][metric_key][ds][algo][seed] = val
                                        except ValueError:
                                            pass
                
                # 2. Process Time & Memory
                for metric, suffix in [('time', '_time'), ('memory', '_memory')]:
                    sheet_name = f"{scen}{suffix}"
                    r_id = sheets.get(sheet_name)
                    if r_id:
                        target_path = rels.get(r_id)
                        full_path = f"xl/{target_path}" if target_path and not target_path.startswith('xl/') else f"xl/worksheets/sheet{r_id}.xml"
                        rows = parse_sheet_data(z, full_path, shared_strings)
                        if rows:
                            header = [c.strip() for c in rows[0]]
                            dataset_idx = next((i for i, c in enumerate(header) if 'dataset' in c.lower()), -1)
                            
                            algos_present = [algo for algo in ['fedavg', 'fedprox', 'scaffold', 'feddyn', 'fedopt', 'pfedme', 'unicsl_static'] if algo in header]
                            algo_indices = {algo: header.index(algo) for algo in algos_present}
                            
                            for row in rows[1:]:
                                if not row or len(row) <= dataset_idx:
                                    continue
                                ds = str(row[dataset_idx]).strip().lower()
                                if ds not in ['cifar10', 'fashionmnist', 'mnist']:
                                    continue
                                    
                                for algo in algos_present:
                                    idx = algo_indices[algo]
                                    if idx < len(row) and row[idx] != '':
                                        try:
                                            val = float(row[idx])
                                            data[scen][metric][ds][algo][seed] = val
                                        except ValueError:
                                            pass
    return data

## 3. Paired t-Test and Table Formatter

In [13]:
def generate_summary_tables(data, unicsl_col="unicsl_static"):
    tables = {}
    algorithms = ['fedavg', 'fedprox', 'scaffold', 'feddyn', 'fedopt', 'pfedme', 'unicsl_static']
    
    for scen in ['iid', 'non_iid', 'non_iid_corrupted']:
        tables[scen] = {}
        for metric in ['accuracy_final', 'accuracy_average', 'time', 'memory']:
            final_rows = []
            
            for ds in ['cifar10', 'fashionmnist', 'mnist']:
                row = {"Dataset": ds}
                
                unicsl_seeds_dict = data[scen][metric][ds][unicsl_col]
                seeds = sorted(list(unicsl_seeds_dict.keys()))
                
                if not seeds:
                    row[unicsl_col] = "-"
                    for algo in algorithms:
                        if algo != unicsl_col:
                            row[algo] = "-"
                    final_rows.append(row)
                    continue
                    
                unicsl_vals = np.array([unicsl_seeds_dict[s] for s in seeds])
                uni_mean = np.mean(unicsl_vals)
                uni_std = np.std(unicsl_vals, ddof=1) if len(unicsl_vals) > 1 else 0.0
                
                row[unicsl_col] = f"{uni_mean:.2f} ± {uni_std:.2f}"
                
                for algo in algorithms:
                    if algo == unicsl_col:
                        continue
                        
                    algo_seeds_dict = data[scen][metric][ds][algo]
                    common_seeds = [s for s in seeds if s in algo_seeds_dict]
                    
                    if len(common_seeds) < 2:
                        row[algo] = "-"
                        continue
                        
                    vals = np.array([algo_seeds_dict[s] for s in common_seeds])
                    u_vals = np.array([unicsl_seeds_dict[s] for s in common_seeds])
                    
                    mean = np.mean(vals)
                    std = np.std(vals, ddof=1) if len(vals) > 1 else 0.0
                    
                    try:
                        if np.allclose(vals, u_vals):
                            t_stat, p_val = 0.0, 1.0
                        else:
                            t_stat, p_val = ttest_rel(vals, u_vals)
                    except Exception:
                        t_stat, p_val = np.nan, np.nan
                        
                    p_text = "<0.001" if (not np.isnan(p_val) and p_val < 0.001) else (f"{p_val:.4f}" if not np.isnan(p_val) else "nan")
                    t_text = f"{t_stat:.2f}" if not np.isnan(t_stat) else "nan"
                    
                    cell_text = (
                        f"{mean:.2f} ± {std:.2f}\n"
                        f"({t_text}, {p_text})"
                    )
                    row[algo] = cell_text
                    
                final_rows.append(row)
            
            cols = ["Dataset"] + [algo for algo in algorithms if algo != unicsl_col] + [unicsl_col]
            df = pd.DataFrame(final_rows)
            for col in cols:
                if col not in df.columns:
                    df[col] = "-"
            tables[scen][metric] = df[cols]
            
    return tables

In [14]:
def generate_ci_tables(data, metric_key="accuracy_average", unicsl_col="unicsl_static"):
    from scipy import stats
    import numpy as np
    import pandas as pd

    scenarios = ['iid', 'non_iid', 'non_iid_corrupted']
    datasets = ['cifar10', 'fashionmnist', 'mnist']
    algorithms = ['fedavg', 'fedprox', 'scaffold', 'feddyn', 'fedopt', 'pfedme']
    
    algo_display_names = {
        'fedavg': 'FedAvg',
        'fedprox': 'FedProx',
        'scaffold': 'SCAFFOLD',
        'feddyn': 'FedDyn',
        'fedopt': 'FedOpt',
        'pfedme': 'pFedMe'
    }

    ci_results = {scenario: {} for scenario in scenarios}
    ci_dfs = {}

    print("=" * 120)
    print(f"MEAN DIFFERENCE AND 95% CONFIDENCE INTERVAL ({metric_key.upper()})")
    print("=" * 120)

    for scenario in scenarios:
        ci_results[scenario] = {}
        print(f"\n{scenario.upper()}")
        print("-" * 120)
        print(f"{'Dataset':<12} {'Algorithm':<12} {'Mean Diff':>12} {'95% CI':>32}")

        rows = []
        for dataset_name in datasets:
            ci_results[scenario][dataset_name] = {}
            row = {"Dataset": dataset_name.upper()}
            
            # UniCSL seed means
            unicsl_seeds_dict = data[scenario][metric_key][dataset_name][unicsl_col]
            seeds = sorted(list(unicsl_seeds_dict.keys()))
            
            if not seeds:
                for algo in algorithms:
                    display_name = algo_display_names[algo]
                    row[display_name] = "-"
                rows.append(row)
                continue
                
            unicsl = np.array([unicsl_seeds_dict[s] for s in seeds])

            for algo in algorithms:
                algo_seeds_dict = data[scenario][metric_key][dataset_name][algo]
                common_seeds = [s for s in seeds if s in algo_seeds_dict]
                
                if len(common_seeds) < 2:
                    display_name = algo_display_names[algo]
                    row[display_name] = "-"
                    ci_results[scenario][dataset_name][algo] = {
                        "mean_difference": np.nan,
                        "ci_lower": np.nan,
                        "ci_upper": np.nan
                    }
                    print(f"{dataset_name:<12} {algo:<12} {'-':>12} {'-':>32}")
                    continue

                unicsl_vals = np.array([unicsl_seeds_dict[s] for s in common_seeds])
                baseline = np.array([algo_seeds_dict[s] for s in common_seeds])

                # Difference = Baseline - UniCSL
                diff = baseline - unicsl_vals

                mean_diff = np.mean(diff)
                n = len(diff)
                se = stats.sem(diff)

                ci_low, ci_high = stats.t.interval(
                    confidence=0.95,
                    df=n-1,
                    loc=mean_diff,
                    scale=se
                )

                ci_results[scenario][dataset_name][algo] = {
                    "mean_difference": mean_diff,
                    "ci_lower": ci_low,
                    "ci_upper": ci_high
                }

                display_name = algo_display_names[algo]
                row[display_name] = (
                    f"{mean_diff:.2f} [{ci_low:.2f}, {ci_high:.2f}]"
                )

                print(
                    f"{dataset_name:<12} "
                    f"{algo:<12} "
                    f"{mean_diff:>10.3f} "
                    f"[{ci_low:.3f}, {ci_high:.3f}]"
                )
            rows.append(row)

        df_ci = pd.DataFrame(rows)
        ci_dfs[scenario] = df_ci

    return ci_dfs

## 4. HTML Exporter

In [15]:
def get_html_report(tables):
    html = """<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Federated Learning Experiment t-Test Summary</title>
    <style>
        body {
            font-family: 'Outfit', 'Inter', -apple-system, sans-serif;
            background-color: #f8fafc;
            color: #1e293b;
            margin: 0;
            padding: 40px 20px;
        }
        .container {
            max-width: 1200px;
            margin: 0 auto;
        }
        h1 {
            color: #0f172a;
            font-size: 2.5rem;
            margin-bottom: 8px;
            font-weight: 700;
        }
        p.subtitle {
            color: #64748b;
            font-size: 1.1rem;
            margin-bottom: 40px;
        }
        .section-card {
            background: white;
            border-radius: 16px;
            box-shadow: 0 4px 6px -1px rgb(0 0 0 / 0.05), 0 2px 4px -2px rgb(0 0 0 / 0.05);
            border: 1px solid #e2e8f0;
            padding: 32px;
            margin-bottom: 48px;
        }
        h2 {
            font-size: 1.8rem;
            color: #0f172a;
            border-bottom: 2px solid #e2e8f0;
            padding-bottom: 12px;
            margin-top: 0;
            margin-bottom: 24px;
            text-transform: capitalize;
        }
        h3 {
            font-size: 1.25rem;
            color: #334155;
            margin-top: 32px;
            margin-bottom: 16px;
        }
        table {
            width: 100%;
            border-collapse: separate;
            border-spacing: 0;
            margin-top: 12px;
            margin-bottom: 24px;
            border-radius: 8px;
            overflow: hidden;
            border: 1px solid #e2e8f0;
        }
        th {
            background-color: #f1f5f9;
            color: #475569;
            font-weight: 600;
            text-align: center;
            padding: 14px 16px;
            font-size: 0.9rem;
            border-bottom: 1px solid #e2e8f0;
            text-transform: uppercase;
        }
        td {
            padding: 16px;
            text-align: center;
            font-size: 0.95rem;
            border-bottom: 1px solid #e2e8f0;
            background-color: #ffffff;
            white-space: pre-line;
            line-height: 1.4;
        }
        tr:last-child td {
            border-bottom: none;
        }
        tr:hover td {
            background-color: #f8fafc;
        }
        .unicsl-highlight {
            background-color: #f0fdf4 !important;
            font-weight: 600;
            color: #166534;
            border-left: 1px solid #bbf7d0;
            border-right: 1px solid #bbf7d0;
        }
        th.unicsl-highlight {
            background-color: #dcfce7 !important;
        }
        .dataset-cell {
            font-weight: bold;
            text-align: left;
            background-color: #fafafa;
            color: #0f172a;
        }
        .notation-guide {
            background-color: #eff6ff;
            border: 1px solid #bfdbfe;
            border-radius: 8px;
            padding: 16px;
            margin-bottom: 32px;
            color: #1e3a8a;
            font-size: 0.95rem;
        }
        .notation-guide ul {
            margin: 8px 0 0 0;
            padding-left: 20px;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>Federated Learning Experiment Results</h1>
        <p class="subtitle">Statistical Analysis and Paired t-Test comparison relative to <strong>UniCSL_Static</strong> (over 5 experimental seeds)</p>
        
        <div class="notation-guide">
            <strong>Table Cell Notation Guide:</strong>
            <ul>
                <li>Values are reported as: <code>Mean ± Standard Deviation</code></li>
                <li>Below each non-proposed method is the paired t-test result: <code>(t-statistic, p-value)</code> calculated against <strong>unicsl_static</strong>.</li>
                <li>A smaller p-value (&lt; 0.05) indicates that the performance difference is statistically significant.</li>
                <li>Sheet abbreviations used in Excel workbook match those in column structure.</li>
            </ul>
        </div>
"""

    for scen in ['iid', 'non_iid', 'non_iid_corrupted']:
        scen_title = scen.replace('_', ' ').upper()
        html += f"""
        <div class="section-card">
            <h2>Scenario: {scen_title}</h2>
        """
        
        for metric, title in [
            ('accuracy_final', 'Final Round Accuracy (Round 50)'),
            ('accuracy_average', 'Average Accuracy (Across 50 Rounds)'),
            ('time', 'Execution Time (Seconds)'),
            ('memory', 'Memory Consumption (MB)')
        ]:
            df = tables[scen][metric]
            html += f"""
            <h3>{title}</h3>
            <table>
                <thead>
                    <tr>
            """
            for col in df.columns:
                col_class = "class='unicsl-highlight'" if col == 'unicsl_static' else ""
                html += f"<th {col_class}>{col}</th>"
            html += """
                    </tr>
                </thead>
                <tbody>
            """
            for _, row in df.iterrows():
                html += "<tr>"
                for col in df.columns:
                    val = row[col]
                    if col == 'Dataset':
                        html += f"<td class='dataset-cell'>{val.upper()}</td>"
                    elif col == 'unicsl_static':
                        html += f"<td class='unicsl-highlight'>{val}</td>"
                    else:
                        html += f"<td>{val}</td>"
                html += "</tr>"
            html += """
                </tbody>
            </table>
            """
            
        html += "</div>"
        
    html += """
    </div>
</body>
</html>
"""
    return html

## 5. Main Execution and Exporting to a Single Excel Workbook

In [16]:
folder = "/Users/mlsilab/Documents/Shuvo/New FL/results/final results for third revision"
print("Reading Excel files and extracting experiment results...")
data = collect_experiment_data(folder)

print("Processing results and running paired t-test analysis...")
tables = generate_summary_tables(data)

print("Calculating Mean Differences and 95% Confidence Intervals for Average Accuracy...")
ci_dfs_average = generate_ci_tables(data, metric_key="accuracy_average")

print("Calculating Mean Differences and 95% Confidence Intervals for Final Round Accuracy...")
ci_dfs_final = generate_ci_tables(data, metric_key="accuracy_final")

print("Exporting results to comparison_results.xlsx...")

# Excel sheets name mapping (must be <= 31 chars)
sheet_name_mapping = {
    ('iid', 'accuracy_final'): 'iid_acc_final',
    ('iid', 'accuracy_average'): 'iid_acc_average',
    ('iid', 'time'): 'iid_time',
    ('iid', 'memory'): 'iid_memory',
    ('non_iid', 'accuracy_final'): 'non_iid_acc_final',
    ('non_iid', 'accuracy_average'): 'non_iid_acc_average',
    ('non_iid', 'time'): 'non_iid_time',
    ('non_iid', 'memory'): 'non_iid_memory',
    ('non_iid_corrupted', 'accuracy_final'): 'non_iid_corr_acc_final',
    ('non_iid_corrupted', 'accuracy_average'): 'non_iid_corr_acc_average',
    ('non_iid_corrupted', 'time'): 'non_iid_corr_time',
    ('non_iid_corrupted', 'memory'): 'non_iid_corr_memory',
}

excel_path = os.path.join(folder, "comparison_results.xlsx")
try:
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        for (scen, metric), sheet_name in sheet_name_mapping.items():
            df = tables[scen][metric]
            df.to_excel(writer, sheet_name=sheet_name, index=False)
            
        # Write CI for Average Accuracy (matching fed_analysis.ipynb)
        for scenario in ['iid', 'non_iid', 'non_iid_corrupted']:
            if scenario == "iid":
                sheet_name = "iid_CI"
            elif scenario == "non_iid":
                sheet_name = "non_iid_CI"
            elif scenario == "non_iid_corrupted":
                sheet_name = "non_iid_corrupted_CI"
            
            df_ci = ci_dfs_average[scenario]
            df_ci.to_excel(writer, sheet_name=sheet_name, index=False)
            
        # Also write CI for Final Round Accuracy
        for scenario in ['iid', 'non_iid', 'non_iid_corrupted']:
            if scenario == "iid":
                sheet_name = "iid_CI_final"
            elif scenario == "non_iid":
                sheet_name = "non_iid_CI_final"
            elif scenario == "non_iid_corrupted":
                sheet_name = "non_iid_corrupted_CI_final"
            
            df_ci = ci_dfs_final[scenario]
            df_ci.to_excel(writer, sheet_name=sheet_name, index=False)
            
    print(f"Successfully generated Excel Workbook with 18 sheets: {excel_path}")
except Exception as e:
    print(f"Error writing Excel: {e}")
    print("Make sure 'openpyxl' is installed: pip install openpyxl")

print("Exporting HTML report...")
html_content = get_html_report(tables)
html_path = os.path.join(folder, "comparison_results.html")
with open(html_path, 'w') as f:
    f.write(html_content)
print(f"Successfully generated HTML Report: {html_path}")

Reading Excel files and extracting experiment results...
Processing results and running paired t-test analysis...
Calculating Mean Differences and 95% Confidence Intervals for Average Accuracy...
MEAN DIFFERENCE AND 95% CONFIDENCE INTERVAL (ACCURACY_AVERAGE)

IID
------------------------------------------------------------------------------------------------------------------------
Dataset      Algorithm       Mean Diff                           95% CI
cifar10      fedavg           -0.856 [-0.980, -0.732]
cifar10      fedprox          -1.008 [-1.130, -0.886]
cifar10      scaffold         -0.836 [-0.952, -0.720]
cifar10      feddyn           -0.274 [-0.400, -0.148]
cifar10      fedopt           -8.190 [-9.204, -7.176]
cifar10      pfedme           -2.206 [-2.450, -1.962]
fashionmnist fedavg           -0.990 [-1.181, -0.799]
fashionmnist fedprox          -1.108 [-1.303, -0.913]
fashionmnist scaffold         -0.930 [-1.177, -0.683]
fashionmnist feddyn           -0.812 [-1.037, -0.587]
fas